## Coupled Mass-Spring System Evolution
This section constructs the time evolution of a coupled mass-spring chain with N = 6 masses and spring constant K = 1. The code uses the provided mass values and initial state vectors Xini and Vini to build the system matrix A, compute a matrix square root of A without relying on the built-in matrix sqrtm, and form the matrix exponentials
$e^{i t \sqrt{A}}$ and $e^{-i t \sqrt{A}}$
for a discrete set of times $t_k = k\Delta t$ with $\Delta t = 0.1$ and $n_t = 1000$. With these propagators, the modal coefficients Xa and Xb are determined so that the solution satisfies the initial conditions, and the position vector X(t) is reconstructed for every time step. The resulting history can be stored in an $N \times n_t$ array for later inspection or plotting.

In [33]:
import numpy as np
import scipy.linalg as linalg
import math as math
import matplotlib.pyplot as plt

In [34]:
dim=6 # number of masses
km=1 # spring constant
M=np.array([[1.85749778], [1.14105586], [1.83143628], [1.79677058], [1.71689508], [1.35939901]]) 
Xini=np.array([[ 0.10257826], [-0.044569 ], [ 0.20937721], [-0.29140314], [-0.21593904], [ 0.23473536]]) 
Vini=np.array([[ 0.96158959], [ 0.82585685], [-0.66922148], [-0.26959786], [ 0.64165591], [ 0.65814696]]) 
nt=1000
epsilon=1e-9

In [35]:
omega=np.zeros((dim))
for i in range (len(omega)):
    omega[i]=(km/M[i])

A=np.zeros((dim,dim))
for i in range(dim):
    for j in range(dim):
        if i==j:
            A[i,j]=2*omega[i]
        if abs(i-j)==1:
            A[i,j]=-omega[i]

C:\Users\Usuario\AppData\Local\Temp\ipykernel_8928\2612930811.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  omega[i]=(km/M[i])


In [36]:
valpa,vecpa=np.linalg.eig(A)
D=vecpa
DInv=np.linalg.inv(vecpa)
Diag=np.zeros((dim,dim))
for i in range(dim):
    Diag[i,i]=np.sqrt(valpa[i])
Asq=D@Diag@DInv

In [37]:
iA=1j*Asq
B=np.linalg.solve(iA,Vini)
Xb=0.5*(Xini-B)
Xa=Xini-Xb
print("Xa",Xa)
print("Xb",Xb)

Xa [[ 0.05128913-0.60867195j]
 [-0.0222845 -0.49332694j]
 [ 0.10468861+0.1449983j ]
 [-0.14570157-0.00682591j]
 [-0.10796952-0.44967065j]
 [ 0.11736768-0.41728258j]]
Xb [[ 0.05128913+0.60867195j]
 [-0.0222845 +0.49332694j]
 [ 0.10468861-0.1449983j ]
 [-0.14570157+0.00682591j]
 [-0.10796952+0.44967065j]
 [ 0.11736768+0.41728258j]]


## Exercise B

In this part we test how quickly the matrix exponential series converges for the normal-mode propagation operator. The code builds an exact exponential matrix from the diagonalized form of $i\sqrt{A}$ and then compares it to partial sums of the Taylor series. The distance between matrices is measured using a norm on the difference, which lets us estimate how many terms are required to reach the selected tolerance.

In [38]:
Aet=np.zeros((dim,dim),dtype=complex)
m1=np.zeros((dim,dim),dtype=complex)
for k in range(dim):
    m1[k,k]=np.exp(1j*Diag[k,k])
Aet=D@m1@DInv

In [39]:
def matrix_trace(M):
    return np.trace(M)

In [40]:
def matrix_distance(M,S):
    return np.linalg.norm(M-S)

In [41]:
def series_expansion(M,k):
    res=np.zeros((dim,dim),dtype=complex)
    for i in range(k):
        res=res+(np.linalg.matrix_power(1j*M,i)/math.factorial(i))
    return res

In [42]:
def indice(Aet,Asq):
    for i in range(40):
        D=matrix_distance(Aet,series_expansion(Asq,i))
        print(np.abs(D))
        if np.abs(D) < epsilon:
            return i-1

In [43]:
indice(Aet,Asq)

2.4600282900997916
2.572710424730444
1.792250580820186
0.8832249790905975
0.3337430381932043
0.10206485380576133
0.026200421960183556
0.005794076681512589
0.0011253808893888898
0.00019486571244788371
3.0439787693314425e-05
4.331173284377568e-06
5.658445345254526e-07
6.833402973015057e-08
7.672142142328721e-09
8.047954409150613e-10


14

## Exercise C

Now convert the time-dependent solution $X(t)$ from the physical coordinate basis into the normal-mode basis. Compute $Y(t) = D^{-1} X(t)$ using the modal transformation matrix D, and store these mode amplitudes in an array with the same time indexing as the original solution. This makes it easy to compare the physical motion of the masses with the behavior of the individual normal modes, and to visualize both representations if desired.

In [44]:
e1=np.zeros((nt,dim,dim),dtype=complex)
e2=np.zeros((nt,dim,dim),dtype=complex)
e=np.round(math.e,15)
for i in range(nt):
    t=0.1*i
    m1=np.zeros((dim,dim),dtype=complex)
    m2=np.zeros((dim,dim),dtype=complex)
    for k in range (dim):
        m1[k,k]=np.exp(1j * Diag[k,k] * t)
        m2[k,k]=np.exp(-1j * Diag[k,k] * t)
    e1[i]=D@m1@DInv
    e2[i]=D@m2@DInv

In [45]:
Xt=np.zeros((nt,dim,1),dtype=complex)
for i in range (nt):
    Xt[i]=e1[i]@Xa+e2[i]@Xb  

In [46]:
Yt=np.zeros((nt,dim,1),dtype=complex)
for i in range (nt):
    Yt[i]=DInv@Xt[i]

In [47]:
print(Xt[500,0])
print(Xt[500,2])
print(Xt[500,4])

[-1.17083848+0.j]
[0.01173514+0.j]
[-0.69076483+0.j]


In [48]:
print(Yt[500,1])
print(Yt[500,3])
print(Yt[500,5])

[0.26660067+0.j]
[-0.01694694+0.j]
[-0.05782493+0.j]
